# GAGA — Group AGgregation enhanced TrAnsformer for Graph Fraud Detection

### Neural Networks — Project Report (Colab notebook)

**Author:** Alessandro Nese — *1990771*  
**Date:** June 2026

---

A clean, dependency-light re-implementation and analysis of **GAGA**,
trained on the **Amazon** and **YelpChi** review graphs for graph-based
fraud detection. The notebook runs end-to-end on a Colab GPU.

## 1. Project Aim & Selected Paper

### Selected paper

> **Label Information Enhanced Fraud Detection against Low Homophily in Graphs**  
> Yuchen Wang, Jinghui Zhang, Zhengjie Huang, Weibin Li, Shikun Feng, Ziheng Ma,
> Yu Sun, Dianhai Yu, Fang Dong, Jiahui Jin, Beilun Wang, Junzhou Luo.  
> *The Web Conference (WWW), 2023.* (`GAGA.pdf` in this folder.)

### What the paper proposes

Graph-based fraud detection is a **node-classification** problem on
multi-relational graphs. Most GNN fraud detectors silently assume **homophily**
(connected nodes share a label) and break down under **low homophily**, which is
exactly the regime fraud lives in: fraudsters *camouflage* by connecting to many
benign nodes, so naive neighbourhood averaging buries the fraud signal in the
benign majority. The paper also notes that **label information** — known to help
node classification — is usually underused (labels are only a supervision signal,
never an input feature).

**GAGA** addresses both with two core ideas:

1. **Group aggregation** — instead of pooling a node's neighbours into one vector,
   split them by label (*benign / fraud / unknown*) and pool each group separately,
   so opposite-class signals stop cancelling out.
2. **Learnable group/hop/relation encodings + a Transformer encoder** — the group
   index *is* the class label, so adding a learnable group encoding injects label
   information directly into the feature space; a vanilla Transformer then
   re-weights the resulting short sequence.

The authors report up to **+24.39%** over competitive fraud detectors on YelpChi,
Amazon and an industrial Baidu dataset.

### Aim of this project

- Re-implement GAGA from scratch with a **light dependency stack** (PyTorch + SciPy,
  *no DGL*), keeping the pipeline transparent and reproducible.
- Reproduce the paper's headline behaviour on the two public datasets (Amazon, YelpChi).
- Analyse the training dynamics, metrics and limitations of the approach.

## 2. Theoretical Background & Key Concepts

### 2.1 Low homophily — why standard GNNs fail

Homophily is measured by the **edge homophily ratio** $h$, the fraction of edges
whose endpoints share a class:

$$ h = \frac{\big|\{(u,v)\in E : y_u = y_v\}\big|}{|E|} $$

High $h\to 1$ (e.g. citation graphs) is what message-passing GNNs are built for:
they **smooth** each node toward its neighbours. Under **low homophily** ($h\to 0$)
this smoothing mixes in other-class features and *dilutes* the signal — a fraud node
averaged toward its mostly-benign neighbourhood becomes indistinguishable from benign.

> **Subtlety (shown in §5).** Because benign nodes dominate (~93% on Amazon), the
> *global* edge homophily looks high. The relevant quantity is **class-conditional**:
> a *fraud* node's neighbours are overwhelmingly benign — that is the camouflage GAGA
> is built to defeat.

### 2.2 Problem setup

A graph $\mathcal{G}(V,E,X,Y)$ with $N$ nodes, $R$ relations (one adjacency per
relation), $d$-dimensional features, and **only a small set of labelled nodes**
(semi-supervised). Labels are binary: $y_v=1$ fraud, $y_v=0$ benign ($C=2$).

GAGA targets two weaknesses of prior work:
- **Challenge 1 — indiscriminate aggregation:** message passing mixes all neighbours
  regardless of label → hurts under low homophily. *Fixed by group aggregation.*
- **Challenge 2 — labels underused:** labels are only supervision, never input.
  *Fixed by the group encoding (label injection).*

### 2.3 The GAGA pipeline (five stages)

**1-2. Group aggregation (weight-free preprocessing, done once).**
Start from a stripped GNN aggregator — just the normalised neighbour sum on raw
features. Then **partition the sum by label**: with $C=2$ the groups are
$V^-$ (benign), $V^+$ (fraud) and $V^*$ (masked/unknown). Each group is pooled
separately:

$$ h_i = \frac{1}{\phi(\cdot)} \sum_{u \in V_i} x_u, \qquad \phi(\cdot)=|V_i|^{\alpha} $$

The target node is masked into $V^*$ to prevent **label leakage**. Running this per
hop $k=1\dots K$ and per relation, then prepending the node's own raw feature $h_v$,
produces a flat input sequence $H_s$ of length

$$ S = R \times (P\times K + 1), \qquad P = C+1 = 3. $$

For Amazon/YelpChi ($R=3, K=2, P=3$): **$S = 3\times(3\cdot 2+1) = 21$ tokens.**

**3. Linear projection.** Each of the $S$ group vectors is lifted from feature dim
$d$ to the Transformer hidden dim $d_H$: $X_s = \sigma(\psi(H_s))$.

**4. Learnable encodings.** A Transformer sees its input as an unordered set, so
three learnable lookup tables stamp each token with *where it came from* — and are
summed elementwise onto $X_s$:
- **Hop encoding** $X_h$ — which hop ($0=$ target, $1\dots K$).
- **Relation encoding** $X_r$ — which relation block.
- **Group encoding** $X_g$ — which label group; **this is the label-injection trick**,
  since the group index equals the class label.

$$ X_{in} = X_s + X_h + X_r + X_g $$

**5. Transformer encoder + readout + prediction.** A vanilla multi-head
self-attention encoder ($L$ layers) re-weights the sequence. GAGA then takes only the
**first (center) token of each relation block** — a BERT-`[CLS]`-style summary that
has absorbed its block — concatenates them across relations into $z_v$, and an MLP
head produces the fraud probability. Training uses (binary) cross-entropy over
labelled nodes only, plus L2 weight decay.

In [ ]:
# (Optional) show the architecture figure from the accompanying notes, if available.
import os
from IPython.display import Image, display
fig = 'GAGA Notion/image.png'
display(Image(fig)) if os.path.exists(fig) else print('Architecture figure not found (skipping).')

## 3. Implementation Details

### 3.1 Datasets

Both graphs ship as MATLAB `.mat` files (features, labels, one sparse adjacency per
relation) and are downloaded straight from the DGL mirror — no DGL dependency, graphs
are built with SciPy sparse matrices. The exact **seed-717** benchmark split is used
(40% train / 50% test / 10% val) so numbers stay comparable to the paper.

| Dataset | Nodes | Feat. dim | Relations | Fraud % | Notes |
|---------|-------|-----------|-----------|---------|-------|
| **Amazon** | 11,944 | 25 | 3 (`U-P-U`, `U-S-U`, `U-V-U`) | ~6.9% | first 3,305 nodes unlabelled, excluded from splits |
| **YelpChi** | 45,954 | 32 | 3 (`R-S-R`, `R-T-R`, `R-U-R`) | ~14.5% | review-spam graph |

Features are **row-normalised**. Both graphs are **low-homophily** (from the fraud
node's perspective), which is the setting GAGA is designed for — visualised in §5.

### 3.2 Graph → sequence (the heart of GAGA)

For every node, `gaga/sequence.py` walks the $K$-hop neighbourhood per relation and
summarises each hop into three group means using **only training labels** (a node
never sees its own label → no leakage):

```
[ center | hop1(benign, fraud, unknown) | hop2(...) ]   x  R relations   ->  (N, 21, d)
```

This is the one expensive, **one-off** step → cached to `seq_cache/` and reused.

### 3.3 Model (`gaga/model.py`)

- `SequenceEncoder`: linear feature projection (+ReLU) followed by three additive
  `nn.Embedding` tables (hop / relation / group), indexed by fixed per-token patterns
  registered as buffers.
- `nn.TransformerEncoder` of `n_layers` standard layers (`batch_first`).
- Readout: take the center token of every relation (stride `1 + K*(C+1)`),
  **concatenate** across relations, classify with a linear head.
- Xavier init; the Amazon model is ~tens of thousands of parameters (printed at train time).

> **Note / deviation from the paper.** This re-implementation uses a 2-logit head with
> `CrossEntropyLoss` (softmax) plus PR-curve **threshold-moving** at test time, rather
> than the paper's single-sigmoid BCE. Behaviour is equivalent for ranking metrics.

### 3.4 Experimental setup

Optimiser **Adam**, early stopping on **validation AUC** (patience 30), best-val
checkpoint saved. Key hyper-parameters per dataset:

| Config | emb_dim | layers | heads | ff_dim | dropout | lr | batch | hops |
|--------|---------|--------|-------|--------|---------|------|-------|------|
| `amazon.json` | 16 | 2 | 4 | 64 | 0.0 | 1e-3 | 256 | 2 |
| `yelp.json`   | 32 | 3 | 4 | 128 | 0.1 | 1e-3 | 512 | 2 |

Shared: `weight_decay=1e-4`, `max_epochs=500`, `early_stop=30`, `seed=717`,
`train_size=0.4`, `val_size=0.1`, features row-normalised.

## 4. Reproducibility & Colab Setup

**Steps:**
1. Set the runtime to GPU: *Runtime → Change runtime type → GPU*.
2. Run the cells below top to bottom — the code is pulled straight from
   **GitHub** ([`Nese2002/GAGA`](https://github.com/Nese2002/GAGA)), so no Google
   Drive copy is required.

**Dependencies:** `torch` (ships with Colab), `scipy`, `scikit-learn`, `matplotlib`,
`tqdm`, `networkx` and `igraph` (for the graph plots) — full list in `requirements.txt`.
The first run downloads the datasets (cached in `data_cache/`) and builds the sequences
(cached in `seq_cache/`); later runs reuse both. The seed-717 split makes results
deterministic.

In [ ]:
# 4.1 Check the GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# 4.2 Dependencies (torch ships with Colab; install the rest)
!pip install -q scipy scikit-learn matplotlib tqdm networkx igraph

In [ ]:
# 4.3 Clone the code from GitHub and put the package on the path.
#     No Google Drive needed — the repo ships gaga/, configs/, train.py, etc.
import sys, os

REPO_URL = 'https://github.com/Nese2002/GAGA.git'
PROJECT_DIR = '/content/GAGA'

if not os.path.isdir(PROJECT_DIR):
    !git clone -q {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull -q

sys.path.append(PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Contents   :', sorted(os.listdir(PROJECT_DIR)))

## 5. Exploratory Visualization of the Graphs

We start with the **whole graph** of each dataset — every node and every edge —
coloured **red = fraud / black = benign** (§5.1). These are large, multi-relational
graphs (~12k / ~46k nodes, millions of edges across 3 relations), so the full view is
intentionally a dense *hairball*; it gives a feel for scale and for how the (few) fraud
nodes are scattered through the benign mass. We then add **targeted** views that tie
directly to GAGA's thesis:

1. **Full graph** — all nodes & edges, fraud (red) vs benign (black).
2. **Homophily** — global vs. *class-conditional*, to expose fraud-node camouflage.
3. **Ego-network** of a single fraud node — the camouflage made visible.
4. **t-SNE** of raw features — how (in)separable the classes are *before* training.
5. **Adjacency spy plots** — sparsity/structure of each relation.

Everything below uses only `gaga.data.load_dataset` (features, labels, per-relation
CSR adjacencies). The per-relation cells (§5.2 onward) use `DATASET` below
(`'amazon'`, fast, or `'yelp'`); the full-graph cell draws **both** datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from gaga.data import load_dataset

DATASET = 'amazon'   # 'amazon' (fast) or 'yelp'
REL_NAMES = {'amazon': ['U-P-U', 'U-S-U', 'U-V-U'],
             'yelp':   ['R-S-R', 'R-T-R', 'R-U-R']}[DATASET]

viz = load_dataset(DATASET, norm_feat=False)   # raw features for honest t-SNE
y = viz['labels']
adjs = [A.tocsr() for A in viz['adjacencies']]

# Labelled-node mask: the first 3305 Amazon nodes are unlabelled placeholders.
labelled = np.ones(len(y), dtype=bool)
if DATASET == 'amazon':
    labelled[:3305] = False
print(f'{DATASET}: nodes={len(y)}  labelled={labelled.sum()}  '
      f'fraud={int(y[labelled].sum())} ({y[labelled].mean():.1%})')

### 5.1 The full graph — every node and every edge

The whole review graph for **both** Amazon and YelpChi, taking the **union of all three
relations** into one undirected graph. We lay it out with `igraph`'s force-directed
solver (fast enough at this scale, where `networkx`'s spring layout is not), draw **all**
edges as faint grey lines, then overlay every node coloured **red = fraud / black =
benign**.

This is deliberately the *hairball* the targeted views below avoid — with millions of
edges it is dense, and the point is exactly that: fraud nodes (red) are a small minority
buried in a benign (black) majority, with no visually obvious clustering. That low,
class-conditional homophily — quantified in §5.2 — is the regime GAGA is built for.

> **Heads-up:** this is the most expensive cell in §5 (layout + rendering of millions of
> edges). Expect roughly a minute or two on a Colab CPU; the per-relation views afterwards
> are cheap.

In [ ]:
import igraph as ig
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy import sparse

def plot_full_graph(name, ax, niter=150, seed=0):
    """Draw the entire graph (union of all relations): all nodes + all edges,
    red = fraud, black = benign."""
    d = load_dataset(name, norm_feat=False)
    yy = d['labels']

    # union of the per-relation adjacencies -> one undirected, simple graph
    A = sum(d['adjacencies']).tocsr()
    A = ((A + A.T) > 0).astype(np.int8)
    A.setdiag(0); A.eliminate_zeros()
    tri = sparse.triu(A, k=1).tocoo()          # each undirected edge once

    # force-directed layout (igraph is C-backed and scales to millions of edges)
    g = ig.Graph(n=A.shape[0], edges=np.column_stack([tri.row, tri.col]))
    np.random.seed(seed)
    P = np.asarray(g.layout_fruchterman_reingold(niter=niter).coords)

    # all edges as one (rasterised) LineCollection, then nodes on top
    segs = np.stack([P[tri.row], P[tri.col]], axis=1)
    ax.add_collection(LineCollection(segs, colors='0.6', linewidths=0.04,
                                     alpha=0.03, rasterized=True, zorder=1))
    benign = yy == 0
    ax.scatter(P[benign, 0],  P[benign, 1],  s=2, c='black', linewidths=0,
               rasterized=True, zorder=2)
    ax.scatter(P[~benign, 0], P[~benign, 1], s=3, c='red',   linewidths=0,
               alpha=0.75, rasterized=True, zorder=3)

    # dense cliques can fling a few nodes far out; clip the view to the bulk
    lo, hi = np.percentile(P, [0.5, 99.5], axis=0)
    pad = 0.05 * (hi - lo)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
    ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'{name}  —  {A.shape[0]:,} nodes, {tri.nnz:,} edges', fontsize=12)
    print(f'[{name}] fraud(red)={int((~benign).sum()):,}  '
          f'benign(black)={int(benign.sum()):,}')

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
plot_full_graph('amazon', axes[0])
plot_full_graph('yelp',   axes[1])

legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
                 markersize=8, label='fraud'),
          Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                 markersize=8, label='benign')]
fig.legend(handles=legend, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Full review graphs (union of all relations) — red = fraud, black = benign',
             fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.97]); plt.show()

### 5.2 Homophily: global vs. class-conditional

The **global** edge homophily $h$ looks high simply because benign nodes dominate
(~93% on Amazon), so benign–benign edges swamp the count. The honest signal is
**class-conditional**: what fraction of a *fraud* node's neighbours are themselves
fraud? If that is low, fraud nodes are camouflaged among benign neighbours — exactly
the low-homophily regime GAGA is designed for.

In [ ]:
rows = []
for name, A in zip(REL_NAMES, adjs):
    coo = A.tocoo()
    m = (coo.row < coo.col) & labelled[coo.row] & labelled[coo.col]
    u, v = coo.row[m], coo.col[m]
    h_global = (y[u] == y[v]).mean()

    # class-conditional: among labelled neighbours, fraction that are fraud
    def frac_fraud_nbrs(node_class):
        nodes = np.where((y == node_class) & labelled)[0]
        tot = pos = 0
        for n in nodes:
            nb = A[n].indices; nb = nb[labelled[nb]]
            pos += int((y[nb] == 1).sum()); tot += len(nb)
        return pos / max(tot, 1)

    rows.append({'relation': name, 'edge_homophily': round(h_global, 3),
                 'fraud_nbrs_of_FRAUD': round(frac_fraud_nbrs(1), 3),
                 'fraud_nbrs_of_BENIGN': round(frac_fraud_nbrs(0), 3)})

import pandas as pd
homo = pd.DataFrame(rows).set_index('relation')
print(homo)

ax = homo[['fraud_nbrs_of_FRAUD', 'fraud_nbrs_of_BENIGN']].plot.bar(figsize=(7, 4))
ax.set_ylabel('fraction of neighbours that are FRAUD')
ax.set_title(f'{DATASET}: a fraud node\'s neighbours are mostly benign (camouflage)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 5.3 Ego-network of a fraud node

One fraud node and its 1-hop neighbourhood under a single relation (capped for
readability), coloured **red = fraud / blue = benign**. The central fraud node
surrounded by blue is the camouflage that defeats plain neighbour-averaging.

In [ ]:
REL = 0                       # which relation to draw
MAX_NBRS = 40                 # cap neighbourhood size for readability
A = adjs[REL]
degs = np.asarray((A > 0).sum(1)).ravel()

# pick a labelled fraud node with a moderate degree so the plot is legible
cands = [n for n in np.where((y == 1) & labelled)[0] if 5 <= degs[n] <= 30]
center = cands[0]
nbrs = A[center].indices
nbrs = nbrs[nbrs != center][:MAX_NBRS]
nodes = np.r_[center, nbrs]

sub = A[nodes][:, nodes]
G = nx.from_scipy_sparse_array(sub)
colors = ['red' if y[n] == 1 else 'royalblue' for n in nodes]
sizes = [220] + [70] * (len(nodes) - 1)

plt.figure(figsize=(7, 6))
nx.draw_spring(G, node_color=colors, node_size=sizes, edge_color='lightgray',
               with_labels=False, seed=0)
plt.title(f'{DATASET} {REL_NAMES[REL]}: ego-net of fraud node {center} '
          f'({int((y[nbrs]==1).sum())}/{len(nbrs)} neighbours are fraud)')
plt.show()

### 5.4 t-SNE of raw node features

A 2-D t-SNE of (a sample of) the raw features, coloured by label. Heavy overlap here
is expected and is the whole point: **raw features alone do not separate fraud from
benign** — the model has to learn it. Re-run this on the learned embeddings $z_v$
after training (§7) for a *before vs. after* comparison, mirroring the paper's Fig. 4.

In [ ]:
from sklearn.manifold import TSNE

SAMPLE = 3000
idx = np.where(labelled)[0]
rng = np.random.RandomState(0)
idx = rng.choice(idx, min(SAMPLE, len(idx)), replace=False)

X = np.asarray(viz['features'])[idx]
emb = TSNE(n_components=2, init='pca', perplexity=30, random_state=0).fit_transform(X)

plt.figure(figsize=(6.5, 6))
for cls, col, lab in [(0, 'royalblue', 'benign'), (1, 'red', 'fraud')]:
    sel = y[idx] == cls
    plt.scatter(emb[sel, 0], emb[sel, 1], s=6, c=col, alpha=0.5, label=lab)
plt.legend(); plt.title(f'{DATASET}: t-SNE of raw features (n={len(idx)})')
plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

### 5.5 Adjacency spy plots

A quick look at the sparsity pattern of each relation's adjacency matrix — relations
differ a lot in density (e.g. on Amazon, `U-S-U` is far denser than `U-P-U`), which is
why GAGA learns a separate **relation encoding** per relation.

In [ ]:
fig, axes = plt.subplots(1, len(adjs), figsize=(13, 4.3))
for ax, name, A in zip(axes, REL_NAMES, adjs):
    ax.spy(A, markersize=0.05, color='black')
    nnz = A.nnz
    ax.set_title(f'{name}  (nnz={nnz:,})')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f'{DATASET}: per-relation adjacency sparsity'); plt.tight_layout(); plt.show()

## 6. Training

### 6.1 Amazon

`train(config)` downloads + builds sequences on the first run (then cached), trains
with early stopping, saves the best model to `checkpoints/amazon_best.pt`, and returns
the test metrics plus the per-epoch history.

In [ ]:
import json
from gaga.trainer import train, train_multiple

with open('configs/amazon.json') as f:
    config = json.load(f)
config

In [ ]:
test_metrics, history = train(config)

### 6.2 YelpChi

Larger graph (~46k nodes); a slightly bigger model (`emb_dim=32`, 3 layers).

In [ ]:
with open('configs/yelp.json') as f:
    yelp_config = json.load(f)

yelp_metrics, yelp_history = train(yelp_config)

## 7. Results & Analysis

### 7.1 Training curves

`plot_history` plots training loss (left axis) against validation AUC / F1-macro
(right axis). A healthy run shows loss decreasing while validation AUC climbs and
plateaus before early stopping triggers.

In [ ]:
from gaga.plot import plot_history
plot_history(history, save_path='checkpoints/amazon_curves.png', show=True)

In [ ]:
plot_history(yelp_history, save_path='checkpoints/yelp_curves.png', show=True)

### 7.2 Metrics table

We report the headline GAGA metrics on the held-out **test** set: **AUC**,
**F1-macro**, **G-Mean**, **AP** (average precision), and the fraud-class
precision/recall. The cell below collects them from both runs into one table.

In [ ]:
import pandas as pd

rows = {'Amazon': test_metrics, 'YelpChi': yelp_metrics}
cols = ['auc', 'f1_macro', 'gmean', 'ap', 'precision_1', 'recall_1', 'f1_fraud']
df = pd.DataFrame(rows).T[cols].round(4)
df.columns = ['AUC', 'F1-macro', 'GMean', 'AP', 'Precision(fraud)', 'Recall(fraud)', 'F1(fraud)']
df

### 7.3 Reference / reproduction targets

Same seed-717 split as the original benchmark; approximate expected results:

| Dataset | AUC (target) | F1-macro (target) |
|---------|-------------|-------------------|
| Amazon  | ~0.96 | ~0.91 |
| YelpChi | ~0.94 | ~0.90 |

### 7.4 Multiple runs (mean ± std)

Because the model is small and the split fixed, variance comes mainly from weight
initialisation and mini-batch order. `train_multiple` repeats training and reports
mean ± std of each metric — a more honest summary than a single run.

In [ ]:
_ = train_multiple(config, n_runs=5)

### 7.5 Interpretation

- **AUC vs F1.** Both datasets are **imbalanced** (Amazon ~6.9% fraud, YelpChi
  ~14.5%), so AUC and F1-macro matter more than raw accuracy. Threshold-moving on the
  validation PR curve is what lifts the fraud-class F1.
- **Why group aggregation helps.** §5 showed a fraud node's neighbours are mostly
  benign; splitting neighbours by label keeps the fraud and benign signals in separate
  tokens instead of averaging them away. The group encoding then lets attention treat
  "this is a fraud-group token" as a first-class feature.
- **Cost profile.** Aggregation is weight-free and cached; the trainable model is a
  small Transformer over only 21 tokens, so training is fast and memory-light
  relative to deep message-passing GNNs.

## 8. Ablation Study — Effect of Each GAGA Component (paper §5.7)

To understand *why* GAGA works, we reproduce the paper's ablation study (§5.7) by
switching individual components on and off and re-training. Each variant is trained
`N_RUNS` times and we report **mean ± std** of the test metrics (the paper uses 5 runs).

The section is organised as the paper's:

- **8.2 Full GAGA** — all components on, the reference point every ablation is compared to.
- **8.3 Group Aggregation** (§5.7.1) — replace label-split group aggregation with a plain
  mean aggregator (the core "low-homophily" mechanism).
- **8.4 Backbone encoder** (§5.7.1) — Transformer vs. a no-attention MLP vs. none, to
  isolate the contribution of self-attention.
- **8.5 Learnable encodings** (§5.7.2, Table 5) — every combination of the group ($X_g$),
  relation ($X_r$) and hop ($X_h$) encodings, with and without group aggregation.
- **8.6 Training rate & label rate** (§5.7.3, Table 6) — sensitivity to how much
  supervision / how many observed labels are available.

> **Compute note.** This is the heaviest part of the notebook. The defaults
> (`ABLATION_DATASET='amazon'`, `N_RUNS=3`) keep it to ~15–30 min on a Colab GPU.
> Switching to `'yelp'`, raising `N_RUNS`, or running §8.6 (which rebuilds the cached
> sequences for every rate) increases this substantially. All variants reuse the
> machinery from `gaga/` — the new switches are `group_agg`, `use_hop/relation/group`,
> `backbone` and `label_rate`.

### 8.1 Setup — controls and helpers

`ABLATION_DATASET` and `N_RUNS` are the two knobs to trade off fidelity vs. runtime.
`run_variants` trains a list of `(label, config-overrides)` variants and returns a tidy
table of mean ± std; `plot_metric` draws the bar chart with error bars.

In [ ]:
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gaga.trainer import train, train_multiple

# ---- ablation controls -------------------------------------------------------
ABLATION_DATASET = 'amazon'   # 'amazon' (fast) or 'yelp' (~46k nodes, slower)
N_RUNS           = 3          # repeats per variant -> mean +/- std (paper uses 5)
ABL_MAX_EPOCHS   = 200        # cap epochs to keep the sweep quick (early stop still applies)

with open(f'configs/{ABLATION_DATASET}.json') as f:
    BASE_CFG = json.load(f)
BASE_CFG['max_epochs'] = ABL_MAX_EPOCHS
print(f"Ablation dataset={ABLATION_DATASET} | runs/variant={N_RUNS} | max_epochs={ABL_MAX_EPOCHS}")

def _slug(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()

def run_variants(variants, n_runs=None, base=None):
    """Train each variant ``n_runs`` times and collect mean +/- std test metrics.

    variants : list of (label, overrides) where overrides patches the base config.
    Returns a DataFrame indexed by label with AUC / AP / F1-macro (+ std) columns.
    """
    n_runs = n_runs or N_RUNS
    base = BASE_CFG if base is None else base
    rows = []
    for label, ov in variants:
        cfg = {**base, **ov, 'run_name': f"abl_{ABLATION_DATASET}_{_slug(label)}"}
        print(f"\n{'='*14} {label} {'='*14}")
        _, s = train_multiple(cfg, n_runs=n_runs)
        rows.append({'variant': label,
                     'AUC': s['auc'][0],      'AUC_std': s['auc'][1],
                     'AP':  s['ap'][0],       'AP_std':  s['ap'][1],
                     'F1_macro': s['f1_macro'][0], 'F1_macro_std': s['f1_macro'][1]})
    return pd.DataFrame(rows).set_index('variant').round(4)

def plot_metric(df, metric='AUC', title=''):
    err = df[f'{metric}_std'] if f'{metric}_std' in df else None
    ax = df[metric].plot.bar(yerr=err, figsize=(max(7, 1.4 * len(df)), 4),
                             capsize=3, color='#4c72b0')
    ax.set_ylabel(f'test {metric}'); ax.set_title(title or metric)
    ax.set_ylim(max(0.0, df[metric].min() - 0.05), min(1.0, df[metric].max() + 0.03))
    plt.xticks(rotation=25, ha='right'); plt.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

### 8.2 Full GAGA — all components (reference)

The complete model: group aggregation **on**, all three learnable encodings **on**,
Transformer backbone. Every later ablation is measured against this row.

In [ ]:
# Full GAGA = base config with no overrides. This is the reference variant.
full_df = run_variants([('GAGA (all components)', {})])
full_df

### 8.3 Effect of Group Aggregation (§5.7.1)

The central idea of GAGA. **With GA**, each hop's neighbours are split by their
(training) label into benign / fraud / unknown group-means, so opposite-class signals
do not cancel. **Without GA** (`group_agg=False`) we fall back to a single mean over all
neighbours — a plain GNN-style aggregator that ignores labels. Everything else (the
Transformer, the relation/hop encodings) is held fixed, so the gap is attributable to
group aggregation. The paper reports a drop of ~3.8% AUC / ~12.9% AP on YelpChi when GA
is removed.

In [ ]:
# group_agg=False rebuilds the sequences with a single mean token per hop.
ga_df = run_variants([
    ('with GA (GAGA)',          {}),
    ('w/o GA (mean aggregator)', {'group_agg': False}),
])
display(ga_df)
plot_metric(ga_df, 'AUC', f'{ABLATION_DATASET}: effect of group aggregation')

### 8.4 Effect of the Backbone Encoder (§5.7.1)

Group aggregation is "effective and portable" — it can be paired with different encoders.
Here we keep GA and all encodings on and swap only the sequence model:

- **Transformer** — GAGA's multi-head self-attention re-weights the group tokens (default).
- **MLP** — a token-wise feed-forward network: same capacity per token but **no attention**,
  so tokens cannot interact.
- **none** — skip the encoder entirely; the center tokens go straight to the linear head.

This isolates how much the *self-attention* contributes on top of group aggregation.
(The paper compares against GCN/GAT/GT backbones; with our DGL-free stack we instead
contrast attention vs. no-attention.)

In [ ]:
# Same GA + encodings; only the backbone changes (no sequence rebuild needed).
backbone_df = run_variants([
    ('Transformer (GAGA)',    {'backbone': 'transformer'}),
    ('MLP (no attention)',    {'backbone': 'mlp'}),
    ('none (linear readout)', {'backbone': 'none'}),
])
display(backbone_df)
plot_metric(backbone_df, 'AUC', f'{ABLATION_DATASET}: effect of the backbone encoder')

### 8.5 Effect of the Learnable Encodings (§5.7.2 — Table 5)

The three additive encodings stamp each token with *where it came from*: the group
($X_g$ = the label-injection trick), the relation ($X_r$) and the hop ($X_h$). This
reproduces the paper's **Table 5**: every subset of the encodings, both **with** group
aggregation and **without** it (where $X_g$ has no meaning and is omitted). The headline
finding is that $X_g$ alone contributes the largest single jump in AUC, confirming that
*injecting the label as a feature* is what drives GAGA.

This is the largest sweep (10 variants × `N_RUNS`); the `w/o GA` rows reuse one rebuilt
sequence cache.

In [ ]:
# Mirrors paper Table 5. Each row toggles a subset of {group, relation, hop}.
enc_variants = [
    # ---- with group aggregation ----
    ('w/ GA: Xg,Xr,Xh', {}),
    ('w/ GA: Xr,Xh',    {'use_group': False}),
    ('w/ GA: Xg',       {'use_relation': False, 'use_hop': False}),
    ('w/ GA: Xr',       {'use_group': False, 'use_hop': False}),
    ('w/ GA: Xh',       {'use_group': False, 'use_relation': False}),
    ('w/ GA: -',        {'use_group': False, 'use_relation': False, 'use_hop': False}),
    # ---- without group aggregation (Xg not applicable) ----
    ('w/o GA: Xr,Xh',   {'group_agg': False}),
    ('w/o GA: Xr',      {'group_agg': False, 'use_hop': False}),
    ('w/o GA: Xh',      {'group_agg': False, 'use_relation': False}),
    ('w/o GA: -',       {'group_agg': False, 'use_relation': False, 'use_hop': False}),
]
enc_df = run_variants(enc_variants)
display(enc_df[['AUC', 'AUC_std', 'AP', 'F1_macro']])
plot_metric(enc_df, 'AUC', f'{ABLATION_DATASET}: learnable-encoding ablation (Table 5)')

### 8.6 Effect of Training Rate & Label Rate (§5.7.3 — Table 6)

Two sensitivity sweeps on how much label information is available:

- **Training rate** — vary `train_size` (40% → 10%): how much *supervision* the model gets.
  Fewer training nodes also means fewer revealed labels, so this is the diagonal of the
  paper's Table 6.
- **Label rate** — fix `train_size=0.4` but reveal only a fraction (`label_rate`) of those
  labels *during group aggregation*, while still supervising on all of them. This isolates
  the value of *observed labels in the aggregation step* (the upper half of Table 6).

GAGA degrades gracefully: even at low rates it stays well above the GNN baselines.

> **Slowest cell in §8:** every rate value rebuilds the cached sequences (group
> aggregation depends on which labels are revealed). Reduce the value lists or `N_RUNS`
> if you are time-constrained.

In [ ]:
def run_rate_sweep(param, values, fixed=None, n_runs=None):
    """Sweep one config parameter (e.g. 'train_size' or 'label_rate') and collect
    mean +/- std of AUC / AP across N_RUNS runs each."""
    n_runs = n_runs or N_RUNS
    fixed = fixed or {}
    rows = []
    for v in values:
        cfg = {**BASE_CFG, **fixed, param: v,
               'run_name': f"abl_{ABLATION_DATASET}_{param}_{int(round(v*100))}"}
        print(f"\n{'='*14} {param}={v} {'='*14}")
        _, s = train_multiple(cfg, n_runs=n_runs)
        rows.append({param: v, 'AUC': s['auc'][0], 'AUC_std': s['auc'][1],
                     'AP': s['ap'][0], 'AP_std': s['ap'][1]})
    return pd.DataFrame(rows).set_index(param).round(4)

train_rate_df = run_rate_sweep('train_size', [0.4, 0.3, 0.2, 0.1])
label_rate_df = run_rate_sweep('label_rate', [0.4, 0.3, 0.2, 0.1], fixed={'train_size': 0.4})
display(train_rate_df)
display(label_rate_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, df, name in [(axes[0], train_rate_df, 'training rate'),
                     (axes[1], label_rate_df, 'label rate')]:
    x = df.index.values * 100
    ax.errorbar(x, df['AUC'], yerr=df['AUC_std'], marker='o', capsize=3, label='AUC')
    ax.errorbar(x, df['AP'],  yerr=df['AP_std'],  marker='s', capsize=3, label='AP')
    ax.set_xlabel(f'{name} (%)'); ax.set_ylabel('test metric')
    ax.set_title(f'{ABLATION_DATASET}: effect of {name}')
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### 8.7 Takeaways

Putting the ablations together (expect the same ordering as the paper, even if absolute
numbers differ slightly from our 2-logit/threshold-moving variant):

- **Group aggregation is the biggest lever** (§8.3): removing it drops AUC and especially
  **AP** the most — it is what defeats the fraud-node camouflage shown back in §5.
- **Self-attention adds a clear margin on top of GA** (§8.4): the Transformer beats the
  no-attention MLP, since attention can re-weight the group tokens per node.
- **The group encoding $X_g$ is the single most valuable encoding** (§8.5): injecting the
  label as a feature is the core trick; relation/hop encodings help but less.
- **GAGA is data-efficient** (§8.6): performance degrades gracefully as training/label
  rates shrink, staying useful even with few observed labels.

Together these confirm the paper's thesis: *how* labels enter the model (group
aggregation + group encoding) matters more than encoder depth.

## 9. Limitations & Reflections

- **Binary, transductive setting.** The formulation assumes $C=2$ and a fixed graph
  with a known split; the group structure ($P=C+1$) would need rethinking for
  multi-class or fully inductive use. (*Generalising GAGA to multi-class is the
  paper's own stated future work.*)
- **Dependence on labelled neighbours.** Group means are computed from *training*
  labels only. When a node has few/no labelled neighbours, its benign/fraud tokens
  fall back to zeros and only the *unknown* group carries signal — quality degrades
  in sparsely-labelled regions (quantified in §8.6).
- **Preprocessing cost & staleness.** Sequences are cached for one split; changing
  hops/split/seed forces a rebuild, and the cache must be invalidated carefully.
- **Implementation deviations.** Softmax 2-logit head + CrossEntropy + threshold
  moving instead of the paper's sigmoid-BCE; metrics are comparable but not bit-exact
  to the reference. The industrial BF10M dataset from the paper is not reproduced.
- **Reflections.** The project shows that *how* labels enter the model can matter more
  than model depth: a shallow Transformer over a cleverly grouped 21-token sequence
  matches deep GNNs in the low-homophily regime. The main engineering lessons were
  avoiding label leakage in the grouping step and making the expensive aggregation
  reproducible via on-disk caching.

## 10. References

**Papers**
1. Y. Wang, J. Zhang, Z. Huang, W. Li, S. Feng, Z. Ma, Y. Sun, D. Yu, F. Dong,
   J. Jin, B. Wang, J. Luo. *Label Information Enhanced Fraud Detection against Low
   Homophily in Graphs.* WWW 2023. (`GAGA.pdf`)
2. A. Vaswani et al. *Attention Is All You Need.* NeurIPS 2017.
3. W. Hamilton, R. Ying, J. Leskovec. *Inductive Representation Learning on Large
   Graphs (GraphSAGE).* NeurIPS 2017.
4. Y. Dou et al. *Enhancing Graph Neural Network-based Fraud Detectors against
   Camouflaged Fraudsters (CARE-GNN).* CIKM 2020. — source of the YelpChi/Amazon graphs.

**Code & datasets**
5. Original GAGA reference implementation — <https://github.com/Orion-wyc/GAGA>
6. YelpChi graph (`FraudYelp.zip`) — <https://data.dgl.ai/dataset/FraudYelp.zip>
7. Amazon graph (`FraudAmazon.zip`) — <https://data.dgl.ai/dataset/FraudAmazon.zip>
8. This project's source: `gaga/` (`data.py`, `sequence.py`, `model.py`,
   `trainer.py`, `metrics.py`, `plot.py`), configs in `configs/`.